In [31]:
import pandas as pd
import glob
import os
import numpy as np
import shutil
from Granger_dictionaries import *

In [35]:
# Grand River
db = "GR"

dir = "..\\data\\granger\\field\\GR\\2023\\Q2"
qrt = "2Q23"
task_code = f"{db}_{qrt}"

idx = 1  # Change this index to select a different file

sampling_company_code = "STRATA"
# Get all Excel file paths in the directory
excel_paths = glob.glob(os.path.join(dir, "*.xlsx"))
# print(excel_paths)

EDD_NAME = excel_paths[idx].split("\\")[-1]
EDD_TO_PROCESS = pd.read_excel(f'{dir}\\{EDD_NAME}', header=None)
# Find coordinates of "Project No.:"
row_idx, col_pos = np.where(EDD_TO_PROCESS == "Project No.:")
# project_no = EDD_TO_PROCESS.iat[row_idx[0], col_pos[0] + 1]
# print(f"Project No.: {project_no}")

print(f"database: {db}")
print(f"quarter: {qrt}")
print(f"file: {EDD_NAME}")
EDD_TO_PROCESS.shape

database: GR
quarter: 2Q23
file: April 2023 SWL_Table 1.xlsx


(17, 7)

In [33]:
cas_dict = {"pH, Field": "pH-F", "Temperature, Field": "Temp-F", "Specific Conductivity, Field": "SpCond-F"}
unit_dict = {"pH-F":"su", "Temp-F": "Deg C", "SpCond-F": "mS/cm"}

# Find first row where column 0 starts with "MW-"

new_columns = ["sys_loc_code", "result_date", "pH, Field", "Temperature, Field", "Specific Conductivity, Field"]
EDD = EDD_TO_PROCESS.copy()

# Keep the real data block and rename columns without using Series.rename()
EDD = EDD_TO_PROCESS.iloc[0:, :5].copy()
EDD.columns = new_columns

# Drop row if no date:
EDD = EDD.dropna(subset=["result_date"])
EDD["sys_loc_code"] = EDD["sys_loc_code"].astype(str).str.strip()

pattern = r"mw-|leachate|sw-|p-|pw|pz|surface|underdrain|cell|lc|electric|recycling|cooper|groesbeck|vmw-"
EDD = EDD[EDD["sys_loc_code"].str.lower().str.contains(pattern, na=False)]

# gets the result value from each field
cols = [
    "pH, Field",
    "Temperature, Field",
    "Specific Conductivity, Field"
]

for col in cols:
    EDD[col] = EDD[col].apply(lambda x: list(str(x).split(","))[-1])

# Unpivot (Melt), the field measurements into the format we need
EDD = EDD.melt(
    id_vars=["sys_loc_code", "result_date"],
    value_vars=[
        "pH, Field",
        "Temperature, Field",
        "Specific Conductivity, Field"
    ],
    var_name="field_parameter_name",
    value_name="result_value"
)

EDD['sample_name'] = EDD.sys_loc_code

# Map to standardized sample names if possible, else keep original code
# EDD['sys_loc_code'] = EDD['sys_loc_code'].map(sample_id_dict).fillna(EDD['sys_loc_code'])  
sample_id_dict_ci = {k.lower(): v for k, v in sample_id_dict.items()}
EDD['sys_loc_code'] = (EDD['sys_loc_code'].str.lower().map(sample_id_dict_ci)).fillna(EDD['sys_loc_code'])

EDD['task_code'] = task_code
EDD['sampling_company_code'] = sampling_company_code
EDD['reportable_result'] = 'YES'
EDD['collection_quarter'] = qrt
# Where 'Result' is empty, set 'detect_flag' to "N" otherwise set it to "Y"
EDD['detect_flag'] = np.where(EDD['result_value'] == '', "N", "Y")
EDD['field_parameter_code'] = EDD['field_parameter_name'].map(cas_dict)
EDD['result_unit'] = EDD['field_parameter_code'].map(unit_dict)

# specific conductivity can be in uS/cm or mS/cm, we need to detect, for now I will use a threshold of 10, below that means mS/cm, above that means uS/cm
result_value = pd.to_numeric(EDD['result_value'], errors='coerce')
EDD['result_unit'] = np.where((EDD['field_parameter_code'] == 'SpCond-F') & (result_value > 99), 'uS/cm', EDD['result_unit'])


EDD['#data_provider'] = sampling_company_code

EDD['value_type'] = "ACTUAL"
EDD['analytic_method'] = "Field"
EDD['sample_matrix_code'] = EDD['sys_loc_code'].apply(assign_matrix)

dates = pd.to_datetime(EDD["result_date"], errors="coerce")
EDD["sys_sample_code"] = np.where(
    dates.notna(),
    EDD["sample_name"].astype(str) + "_" + dates.dt.strftime("%Y%m%d"),
    np.nan
)

# final cleanup remove "#" symbols from sys_sample_code only
EDD["sys_sample_code"] = EDD["sys_sample_code"].str.replace("#", "", regex=False)

In [34]:
# 1. path to template
template_path = r"..\EDDTemplate\Field EDD Template.xlsx"
template_df = pd.read_excel(template_path, sheet_name="FieldResults_v1")

EDD = EDD.reindex(columns=template_df.columns)
expected_columns = ["#data_provider", "sys_loc_code", "sys_sample_code", "field_parameter_code", "field_parameter_name", "result_date", "result_time", "start_depth", "end_depth", "depth_unit", "result_value", "result_unit", "quantitation_limit", "task_code", "sample_matrix_code", "qualifier", "sampling_company_code", "sampling_reason", "sample_method", "reportable_result", "value_type", "remark", "detect_flag", "collection_quarter", "sample_name", "lab_name_code", "analytic_method", "validator_qualifiers", "fraction", "custom_field_4", "TRC_Project_Number", "sampler"]

try:
    assert EDD.columns.tolist() == expected_columns
    print("Column names match the expected list.")
except AssertionError:
    print("Column names do not match the expected list.")
    # The EDD to process has extra columns that are not in the expected list. Print the extra columns.
    print("Extra columns in EDD to process:", set(template_df.columns.tolist()) - set(expected_columns
    ))

# 2. Make an exact copy of the template file
# Ouptut path for the new EDD file to be created
# Make a unique name for the output file by including the original file name and a timestamp
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
output_path = f"{dir}\\{EDD_NAME.split('.')[0]}_FIELD_EDD_{timestamp}.xlsx"
shutil.copy(template_path, output_path)

# 3. Write your dataframe into the copied template
# (Using 'a' mode allows you to append/write data to an existing sheet)
with pd.ExcelWriter(
    output_path, engine="openpyxl", mode="a", if_sheet_exists="overlay"
) as writer:
    EDD.to_excel(
        writer,
        sheet_name="FieldResults_v1",  # <-- Change this to the exact name of the sheet in your template
        index=False,
        header=False,  # <-- Set to False if your template already has the headers typed out
        startrow=2,  # <-- Starts writing on row 2 (0-indexed), assuming row 1 has your headers
    )

Column names do not match the expected list.
Extra columns in EDD to process: set()


c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\ageglio-1\EDD Data Processor\dataprocessor\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
